In [0]:
# CELL 1: Setup & Load
# -------------------------------------------------------------------------
from pyspark.sql.functions import col, to_date, when, lit

# 1. Define Azure Config (PASTE YOUR KEYS if needed)
storage_account_name = ""
storage_account_key = ""
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# --- CRITICAL FIX: Auto-create the 'silver' container if missing ---
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# 2. Define Paths
base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"

# 3. Read Bronze CSVs
print("Reading Bronze files...")
df_trans_bronze = spark.read.option("header", "true").csv(f"{base_path}/transactions/landing/current.csv")
df_proj_bronze = spark.read.option("header", "true").csv(f"{base_path}/projects/landing/current.csv")
df_units_bronze = spark.read.option("header", "true").csv(f"{base_path}/units/landing/current.csv")
 
print("✅ Bronze Data Loaded & Silver Container Configured.")

In [0]:
# CELL 2: Clean Projects Dimension (Status & Dates Only)
# -------------------------------------------------------------------------

df_projects_clean = df_proj_bronze.select(
    col("project_number").cast("int"),
    col("project_status"),
    col("percent_completed").cast("double"),
    # FIX: Date format set to 'dd-MM-yyyy'
    to_date(col("project_start_date"), "dd-MM-yyyy").alias("start_date"),
    to_date(col("completion_date"), "dd-MM-yyyy").alias("completion_date"),
    col("master_project_en").alias("master_community")
).dropDuplicates(["project_number"])

print("✅ Projects Cleaned.")
display(df_projects_clean.limit(5))

In [0]:
# CELL 3: Clean Transactions Fact
# -------------------------------------------------------------------------
df_trans_clean = df_trans_bronze.select(
    col("transaction_id"),
    
    # FIX: Date format set to 'dd-MM-yyyy'
    to_date(col("instance_date"), "dd-MM-yyyy").alias("transaction_date"),
    
    col("procedure_name_en").alias("transaction_type"),
    col("trans_group_en").alias("transaction_group"),
    
    # Location & Project Info
    col("area_name_en").alias("area_name"),
    col("building_name_en").alias("building_name"),
    col("project_number").cast("int"),
    
    # --- We get the English Name from here ---
    col("project_name_en").alias("project_name"),
    col("nearest_landmark_en").alias("landmark"),
    
    # Property Details
    col("property_type_en").alias("property_type"),
    col("property_sub_type_en").alias("property_subtype"),
    col("rooms_en").alias("room_type"), # e.g., "1 B/R"
    
    # Metrics
    col("procedure_area").cast("double").alias("area_sqft"),
    col("actual_worth").cast("double").alias("price"),
    col("meter_sale_price").cast("double")
)

# Filter garbage data
df_trans_clean = df_trans_clean.filter(col("price") > 0)
 
print("✅ Transactions Cleaned.")
display(df_trans_clean.limit(5))

In [0]:
# CELL 4: Join & Enrich (Silver Master Table)
# -------------------------------------------------------------------------

# 1. Join Transactions with Projects
df_enriched = df_trans_clean.join(
    df_projects_clean,
    on="project_number",
    how="left"
)

# 2. Enrich: Calculate Price Per SqFt
df_enriched = df_enriched.withColumn(
    "price_per_sqft",
    when(col("area_sqft") > 0, col("price") / col("area_sqft")).otherwise(0)
)

# 3. Enrich: Is Off-Plan?
df_enriched = df_enriched.withColumn(
    "is_offplan_flag",
    when(
        (col("transaction_date") < col("completion_date")) | (col("percent_completed") < 100),
        lit(True)
    ).otherwise(lit(False))
)

print("✅ Data Enriched & Joined.")
display(df_enriched.select("transaction_date", "project_name", "room_type", "price", "is_offplan_flag").limit(10))

In [0]:
# CELL 5: Write to Silver Layer (Physical Files)
# -------------------------------------------------------------------------
# 1. Write the Master Transactions Table
print("Writing Transactions to Silver...")
df_enriched.write.format("delta").mode("overwrite").save(f"{silver_path}/transactions")

# 2. Write the Clean Units Table
print("Writing Units to Silver...")
df_units_clean = df_units_bronze.select(
    col("property_id"), # Unique Key
    col("project_id").cast("int").alias("project_number"), # Rename to match other tables
    col("unit_number"),
    
    # Metrics
    col("rooms").alias("bedrooms"), # Numeric (e.g., "1")
    col("rooms_en").alias("room_type"),# Text (e.g., "1 B/R")
    
    # --- FIX: Use 'actual_area' ---
    col("actual_area").cast("double").alias("unit_area")
).dropDuplicates(["property_id"]) # Deduplicate
df_units_clean.write.format("delta").mode("overwrite").save(f"{silver_path}/units")

print("🎉 SUCCESS: Silver Layer Created (Delta Format).")

In [0]:
# CELL 6: Verification (Check Data Exists)
# -------------------------------------------------------------------------
print("🔍 Verifying Silver Layer...")
 
try:
    # 1. Load the new Delta Tables directly from the file path
    df_silver_trans = spark.read.format("delta").load(f"{silver_path}/transactions")
    df_silver_units = spark.read.format("delta").load(f"{silver_path}/units")
    
    # 2. Print Counts
    count_trans = df_silver_trans.count()
    count_units = df_silver_units.count()
    
    print(f"✅ VERIFIED: Transactions Table contains {count_trans} rows.")
    print(f"✅ VERIFIED: Units Table contains {count_units} rows.")
 
    # 3. Display the data (This is your visual check)
    if count_trans > 0:
        print("🚀 Success! Here is a preview of your clean Silver data:")
        display(df_silver_trans.limit(5))
    else:
        print("⚠️ WARNING: Tables exist but are empty.")
 
except Exception as e:
    print(f"❌ ERROR: Could not read Silver tables. {e}")